In [ ]:
import pandas as pd
import re
from datetime import datetime
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_columns', None)

In [254]:
bein_file = r"S:\05.08.25\26154360_BEINDATANEWRPT.CSV"
osn_file = r"S:\hawary bein\Hawary OSN\OSN.xlsx"
today = datetime.now()
after3months = today + relativedelta(months=3)
yearinpast = today - relativedelta(years=1)
print (today)
print(after3months)
print(yearinpast)

2025-08-05 13:40:20.096746
2025-11-05 13:40:20.096746
2024-08-05 13:40:20.096746


In [ ]:

def is_valid_phone(p):
    if pd.isna(p):
        return False
    p = str(p)
    # Remove spaces, dashes, parentheses etc.
    cleaned = re.sub(r'[^\d+]', '', p)
    
    # Check if starts with +, 00, or 0
    if cleaned.startswith('+') or cleaned.startswith('00') or cleaned.startswith('0'):
        # Remove non-digits to count digits only
        digits_only = re.sub(r'\D', '', cleaned)
        # Require at least 10 digits
        return len(digits_only) >= 10
    return False


def is_valid_email(m: str) -> bool:
    if not isinstance(m, str):
        return False
    email_pattern = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
    return re.match(email_pattern, m) is not None

beIN

In [ ]:
data = pd.read_csv(bein_file,dtype='str')

sub_types = ['beIN Quartar Installment', 'CNE Subscriber',
            'BeIN sports CC', 'beIN Bi Installment', 
            'Bein NC', 'beIN Installment Sub']

data = data.loc[data['Customer Type'].isin(sub_types)]
data = data.loc[~data['Contract Number'].isin(['3532916','2098194'])]

In [ ]:
data['End Date'] = pd.to_datetime(data['End Date'],dayfirst=True)
data = data[~data['Plan'].str.contains('ART',na=False)]
data = data[data['Plan']!='Temp01']

In [ ]:
bein_active = data.loc[data['Status']=='Active']
bein_active = bein_active.sort_values(['End Date'],ascending=False)
bein_active = bein_active.drop_duplicates(subset=['Customer Number'],keep='first')
bein_active = bein_active.loc[:,['Customer Number','Customer Type','Smart Card','Decoder']]
bein_active.to_excel("hawary_bein_active.xlsx", index=False)

In [ ]:
bein_active_to_disconnect_3months = data.loc[(data['Status']=='Active') & (data['End Date']<after3months)]
bein_active_to_disconnect_3months = bein_active_to_disconnect_3months.sort_values(['End Date'],ascending=False)
bein_active_to_disconnect_3months = bein_active_to_disconnect_3months.loc[:,['Customer Number','Customer Type','Smart Card','Decoder']]
bein_active_to_disconnect_3months = bein_active_to_disconnect_3months.drop_duplicates(subset=['Customer Number'],keep='first')
bein_active_to_disconnect_3months.to_excel('hawary_bein_active_to_disconnect_3months.xlsx',index=False)

In [ ]:
bein_suspended = data.loc[(data['Status']=='DIS') & (data['End Date']>today)]
bein_suspended = bein_suspended.sort_values(['End Date'],ascending=False)
bein_suspended = bein_suspended.loc[:,['Customer Number','Customer Type','Smart Card','Decoder']]
bein_suspended = bein_suspended.drop_duplicates(subset=['Customer Number'],keep='first')
bein_suspended = bein_suspended.loc[~bein_suspended['Customer Number'].isin(bein_active['Customer Number'])]
bein_suspended.to_excel('hawary_bein_suspended.xlsx',index=False)

In [ ]:
bein_disconnected_year_plus = data.loc[(data['Status']=='DIS') & (data['End Date']<=yearinpast)]
bein_disconnected_year_plus = bein_disconnected_year_plus.sort_values(['End Date'],ascending=False)
bein_disconnected_year_plus = bein_disconnected_year_plus.loc[:,['Customer Number','Customer Type','Smart Card','Decoder']]
bein_disconnected_year_plus = bein_disconnected_year_plus.drop_duplicates(subset=['Customer Number'],keep='first')
bein_disconnected_year_plus = bein_disconnected_year_plus.loc[~bein_disconnected_year_plus['Customer Number'].isin(bein_active['Customer Number'])]
bein_disconnected_year_plus.to_excel('hawary_bein_disconnected_year_plus.xlsx',index=False)

In [ ]:
#data.loc[data['Customer Number']=='13376476']

OSN

In [ ]:
osn = pd.read_excel(osn_file, dtype='str')
osn_sub_types=[
'DTH_CC_IP1',
'DTH_CASH_IP1',
'DTH_CASH_IP12',
'DTH_CASH_IP14',
'DTH_CASH_IP3',
'DTH_CASH_IP6',
'DTH_CASH_IP7',
'DTH_CC_IP12',
'DTH_CC_IP14',
'DTH_CC_IP3',
'DTH_CC_IP6',
'DTH_CC_IP7',
'OSN DTH'
]

osn = osn.loc[osn['CUSTOMER_TYPE'].isin(osn_sub_types)]

email_validator = osn['EMAIL'].apply(is_valid_email)
mobile1_validator = osn['MOBILE1'].apply(is_valid_phone)
mobile2_validator = osn['MOBILE2'].apply(is_valid_phone)
fax_validator = osn['FAX'].apply(is_valid_phone)

osn= osn.loc[email_validator | mobile1_validator | mobile2_validator | fax_validator]
osn['NEXT_INVOICE_DATE'] = pd.to_datetime(osn['NEXT_INVOICE_DATE'], format='%Y-%m-%d %H:%M:%S')

In [ ]:
osn_active = osn.loc[osn['STATUS']=='Active']
osn_active = osn_active.drop_duplicates(subset=['CUSTOMER_NUMBER'], keep='first')
osn_active = osn_active.loc[:,['CUSTOMER_NUMBER','CUSTOMER_TYPE','EMAIL','MOBILE1','MOBILE2','FAX']]
osn_active.to_excel("hawary_osn_active.xlsx",index=False)

In [ ]:
osn_active_to_disconnect_3months = osn.loc[(osn['STATUS']=='Active') & (osn['NEXT_INVOICE_DATE']<after3months)]
osn_active_to_disconnect_3months = osn_active_to_disconnect_3months.drop_duplicates(subset=['CUSTOMER_NUMBER'], keep='first')
osn_active_to_disconnect_3months = osn_active_to_disconnect_3months.loc[:,['CUSTOMER_NUMBER','CUSTOMER_TYPE','EMAIL','MOBILE1','MOBILE2','FAX']]
osn_active_to_disconnect_3months.to_excel("hawary_osn_active_to_disconnect_3months.xlsx",index=False)

In [ ]:
osn_suspended = osn.loc[(osn['STATUS']=="Inactive") & (osn['NEXT_INVOICE_DATE']>today)]
osn_suspended = osn_suspended.sort_values(['NEXT_INVOICE_DATE'],ascending=False)
osn_suspended = osn_suspended.drop_duplicates(subset=['CUSTOMER_NUMBER'], keep='first')
osn_suspended = osn_suspended.loc[~osn_suspended['CUSTOMER_NUMBER'].isin(osn_active['CUSTOMER_NUMBER'])]
osn_suspended = osn_suspended.loc[:,['CUSTOMER_NUMBER','CUSTOMER_TYPE','EMAIL','MOBILE1','MOBILE2','FAX']]
osn_suspended.to_excel("hawary_osn_suspended.xlsx",index=False)

In [ ]:
osn_disconnected_year_plus = osn.loc[(osn['STATUS']=='Inactive') & (osn['NEXT_INVOICE_DATE']<=yearinpast)]
osn_disconnected_year_plus = osn_disconnected_year_plus.sort_values(['NEXT_INVOICE_DATE'],ascending=False)
osn_disconnected_year_plus = osn_disconnected_year_plus.drop_duplicates(subset=['CUSTOMER_NUMBER'], keep='first')
osn_disconnected_year_plus = osn_disconnected_year_plus.loc[~osn_disconnected_year_plus['CUSTOMER_NUMBER'].isin(osn_active['CUSTOMER_NUMBER'])]
osn_disconnected_year_plus = osn_disconnected_year_plus.loc[:,['CUSTOMER_NUMBER','CUSTOMER_TYPE','EMAIL','MOBILE1','MOBILE2','FAX']]
osn_disconnected_year_plus.to_excel("hawary_osn_disconnected_year_plus.xlsx",index=False)

In [ ]:
osn.loc[osn['CUSTOMER_NUMBER']=='5915590']